# E1 — ExtraSensory Ingest + User Split
**Isolated enrichment-experiment track.** Reads `data/enrichment_experiment/`, writes
`results/enrichment_experiment/`. Does not touch the health_trajectory pipeline (01-04/03b).

Steps: load all 60 users → attach the official 5-fold **by-user** split (leakage check) →
label / feature coverage → save a manifest.

In [1]:
import pandas as pd
import ee_common as ee

ee.ensure_output_dirs()
print("repo root :", ee.REPO_ROOT)
print("features  :", ee.FEATURES_DIR, "| exists:", ee.FEATURES_DIR.exists())
print("results   :", ee.RESULTS_DIR)
print("n user files:", len(ee.uuid_files()))

repo root : C:\Project\Apple Health Data
features  : C:\Project\Apple Health Data\data\enrichment_experiment\extrasensory\features_and_labels | exists: True
results   : C:\Project\Apple Health Data\results\enrichment_experiment
n user files: 60


## 1. Load all users

In [2]:
df = ee.load_all()                      # all 60 users, adds a 'uuid' column
feat = ee.feature_cols(df)
labs = ee.label_cols(df)
print(f"rows={len(df):,}  cols={df.shape[1]}  users={df['uuid'].nunique()}")
print(f"features={len(feat)}  label cols={len(labs)}  (+ uuid, timestamp, label_source)")
df.head(3)

rows=377,346  cols=279  users=60
features=225  label cols=51  (+ uuid, timestamp, label_source)


,uuid,timestamp,raw_acc:magnitude_stats:mean,raw_acc:magnitude_stats:std,raw_acc:magnitude_stats:moment3,raw_acc:magnitude_stats:moment4,raw_acc:magnitude_stats:percentile25,raw_acc:magnitude_stats:percentile50,raw_acc:magnitude_stats:percentile75,raw_acc:magnitude_stats:value_entropy,...,label:STAIRS_-_GOING_DOWN,label:ELEVATOR,label:OR_standing,label:AT_SCHOOL,label:PHONE_IN_HAND,label:PHONE_IN_BAG,label:PHONE_ON_TABLE,label:WITH_CO-WORKERS,label:WITH_FRIENDS,label_source
0,00EABED2-271D-49D8-B599-1D4A09240601,1444079161,0.996815,0.003529,-0.002786,0.006496,0.995203,0.996825,0.998502,1.748756,...,NaN,NaN,0.0,NaN,NaN,NaN,1.0,1.0,NaN,2
1,00EABED2-271D-49D8-B599-1D4A09240601,1444079221,0.996864,0.004172,-0.003110,0.007050,0.994957,0.996981,0.998766,1.935573,...,NaN,NaN,0.0,NaN,NaN,NaN,1.0,1.0,NaN,2
2,00EABED2-271D-49D8-B599-1D4A09240601,1444079281,0.996825,0.003667,0.003094,0.006076,0.994797,0.996614,0.998704,2.031780,...,NaN,NaN,0.0,NaN,NaN,NaN,1.0,1.0,NaN,2


In [3]:
# rows per user
per_user = (df.groupby("uuid").size().rename("n_samples")
              .sort_values(ascending=False))
print("samples/user  min/median/max:",
      per_user.min(), int(per_user.median()), per_user.max())
per_user.describe()

samples/user  min/median/max: 1600 6312 11996


count       60.000000
mean      6289.100000
std       2482.909684
min       1600.000000
25%       3947.750000
50%       6312.500000
75%       8177.750000
max      11996.000000
Name: n_samples, dtype: float64

## 2. Official 5-fold split (by user — no leakage)

In [4]:
folds = ee.load_folds()
rows = []
for k, s in folds.items():
    leak = set(s["train"]) & set(s["test"])
    rows.append({"fold": k, "n_train_users": len(s["train"]),
                 "n_test_users": len(s["test"]), "user_leak": len(leak)})
fold_tbl = pd.DataFrame(rows)
assert fold_tbl["user_leak"].sum() == 0, "USER LEAKAGE between train/test!"
union = set().union(*[set(s["train"]) | set(s["test"]) for s in folds.values()])
print("every fold leak-free; union of users across folds:", len(union))
fold_tbl

every fold leak-free; union of users across folds: 60


,fold,n_train_users,n_test_users,user_leak
0,0,48,12,0
1,1,48,12,0
2,2,48,12,0
3,3,48,12,0
4,4,48,12,0


In [5]:
# which fold each user is a TEST user in (each user tests exactly once in 5-fold CV)
test_of = {}
for k, s in folds.items():
    for u in s["test"]:
        test_of[u] = k
manifest = (per_user.reset_index()
            .assign(test_fold=lambda d: d["uuid"].map(test_of)))
print("users missing a test fold:", manifest["test_fold"].isna().sum())
manifest.head()

users missing a test fold: 0


,uuid,n_samples,test_fold
0,78A91A4E-4A51-4065-BDA7-94755F0BB3BB,11996,2
1,86A4F379-B305-473D-9D83-FC7D800180EF,10738,3
2,9759096F-1119-4E19-A0AD-6F16989C7E1C,9959,2
3,7CE37510-56D0-4120-A1CF-0E23351428D2,9761,2
4,9DC38D04-E82E-4F29-AB52-B476535226F2,9686,2


## 3. Label & feature coverage

In [6]:
# self-reported labels are sparse: fraction of samples where each label is observed (not NaN)
lab_obs = df[labs].notna().mean().sort_values(ascending=False)
print("label observation rate (share of samples with a non-NaN value):")
print("  highest:"); print(lab_obs.head(5).to_string())
print("  lowest :"); print(lab_obs.tail(5).to_string())

label observation rate (share of samples with a non-NaN value):
  highest:
label:LOC_home       0.941666
label:FIX_walking    0.812501
label:SITTING        0.812501
label:OR_standing    0.812501
label:LYING_DOWN     0.804893
  lowest :
label:STROLLING     0.142471
label:LAB_WORK      0.130901
label:AT_THE_GYM    0.117645
label:AT_A_BAR      0.088036
label:SINGING       0.066562


In [7]:
# feature missingness (features are the ML/LLM input; note how much is NaN)
feat_na = df[feat].isna().mean()
print(f"feature NaN rate  mean={feat_na.mean():.1%}  "
      f"max={feat_na.max():.1%}  (>50% NaN: {(feat_na>0.5).sum()} feats)")

feature NaN rate  mean=15.7%  max=93.1%  (>50% NaN: 16 feats)


## 4. Save manifest

In [8]:
manifest.to_csv(ee.RESULTS_DIR / "e1_user_manifest.csv", index=False)
(lab_obs.rename("obs_rate").reset_index()
        .rename(columns={"index": "label"})
        .to_csv(ee.RESULTS_DIR / "e1_label_observation_rate.csv", index=False))
print("saved:")
print("  ", ee.RESULTS_DIR / "e1_user_manifest.csv")
print("  ", ee.RESULTS_DIR / "e1_label_observation_rate.csv")

saved:
   C:\Project\Apple Health Data\results\enrichment_experiment\e1_user_manifest.csv
   C:\Project\Apple Health Data\results\enrichment_experiment\e1_label_observation_rate.csv


**E1 done.** 60 users loaded, official by-user 5-fold split verified leak-free, coverage
summarised. Next: **E2** turns the raw 51 labels into the fixed 3-field context schema.